# Experiment 6 Cross-Model Replication: SmolLM2 1.7B Instruct

**Phase A: Verification only**

This notebook is a cross-model replication of the frozen Experiment 6 protocol. The cases, prompts, order, scoring, and deterministic decoding policy are unchanged; only the model is changed.

It implements the frozen:

1. Experiment 6: Blinded Test Set v1
2. Experiment 6: Verification Architecture v1
3. Experiment 6: Exact Verification and Scoring Protocol v1

It compares:

- **HV**: holistic verification
- **DV**: decomposed verification
- **Mechanical DV**: deterministic composition of DV component judgments

The notebook saves raw outputs before scoring. It contains no repair phase and no behavioral probes.


In [ ]:
# Install dependencies
!pip -q install "transformers==4.46.3" "accelerate==1.0.1" pandas


In [ ]:
# Imports and deterministic configuration
import os, re, json, random, time, hashlib
from pathlib import Path
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "HuggingFaceTB/SmolLM2-1.7B-Instruct"
SEED = 42
MAX_NEW_TOKENS_HV = 220
MAX_NEW_TOKENS_DV = 280

random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)
if DEVICE != "cuda":
    raise RuntimeError("GPU required for the frozen Experiment 6 run. In Colab select Runtime > Change runtime type > GPU.")


In [ ]:
# Mount Google Drive and create output directory
from google.colab import drive
drive.mount("/content/drive")

OUT_DIR = Path("/content/drive/MyDrive/Sarah_Memory_Experiment_6_SmolLM2_Replication")
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("Outputs:", OUT_DIR)


In [ ]:
# Frozen prompts

HV_PROMPT = '''You are evaluating whether an AI's current interpretation of a person is justified by the human evidence provided.

Consider the exact meaning of the evidence, the order of statements when more than one statement is present, and the context in which each statement applies.

Decide whether the candidate interpretation is justified as a current interpretation of the person.

Do not accept an interpretation merely because it is plausible. Do not infer motives, traits, preferences, causes, or general patterns beyond what the evidence supports.

Return exactly:

DECISION: <ACCEPT|REJECT>
EVIDENCE: <comma separated evidence IDs>
JUSTIFICATION: <one concise explanation>'''

DV_PROMPT = '''You are evaluating whether an AI's current interpretation of a person is justified by the human evidence provided.

Evaluate the candidate independently on three dimensions.

ENTAILMENT: Does the human evidence justify every substantive claim made by the candidate interpretation? Do not treat plausibility as evidence.

TEMPORAL VALIDITY: Considering the order of relevant evidence, is the candidate valid as the current interpretation?

SCOPE: Is the candidate no broader in people, situations, contexts, causes, preferences, or generality than the evidence warrants?

Perform all three checks for every case. Do not assume which dimension, if any, contains a problem.

Then make your own final judgment about whether the candidate should be accepted as a current interpretation.

Return exactly:

ENTAILMENT: <PASS|FAIL|N/A>
TEMPORAL_VALIDITY: <PASS|FAIL|N/A>
SCOPE: <PASS|FAIL|N/A>
EVIDENCE: <comma separated evidence IDs>
FINAL_DECISION: <ACCEPT|REJECT>
JUSTIFICATION: <one concise explanation>'''


In [ ]:
# Frozen 16-case blinded test set.
# Neutral evidence only: ID, turn, exact human statement.

CASES = [
    {"id":"S1","evidence":[("E1",1,"When you explain technical topics to me, examples usually help me understand them.")],
     "candidate":"Examples are useful when explaining technical topics to Sarah."},
    {"id":"S2","evidence":[("E1",1,"For large conferences, I often wait until I know what the travel will cost.")],
     "candidate":"Travel cost can be relevant to Sarah's decisions about large conferences."},
    {"id":"S3","evidence":[("E1",1,"For quick work messages, I prefer the main point first.")],
     "candidate":"Sarah prefers the main point first in quick work messages."},
    {"id":"S4","evidence":[("E1",1,"I postponed today's client meeting because my train was cancelled.")],
     "candidate":"Sarah postponed today's client meeting because her train was cancelled."},
    {"id":"S5","evidence":[("E1",1,"When I'm comparing software for a project, two or three strong options are more useful than a huge list.")],
     "candidate":"Sarah finds a small set of strong options useful when comparing software for a project."},
    {"id":"S6","evidence":[("E1",1,"I still want to finish the documentary. I'm pausing it this month because paid work needs my attention.")],
     "candidate":"Sarah remains interested in finishing the documentary but is currently prioritizing paid work."},
    {"id":"S7","evidence":[("E1",1,"Unexpected experimental results are more useful to me when we consider several possible explanations before choosing one.")],
     "candidate":"Sarah prefers considering multiple explanations for unexpected experimental results before settling on one."},
    {"id":"S8","evidence":[("E1",1,"I usually prefer written preparation before an important professional meeting.")],
     "candidate":"Written preparation is generally useful to Sarah before important professional meetings."},

    {"id":"E1","evidence":[("E1",1,"I've postponed working on the screenplay again.")],
     "candidate":"Sarah is losing motivation for her screenplay."},
    {"id":"E2","evidence":[("E1",1,"I'm waiting for the technical evaluation before deciding whether to join the project.")],
     "candidate":"Sarah is uncertain whether the project aligns with her career goals."},

    {"id":"T1","evidence":[
        ("E1",1,"I'm not attending the conference because the ticket is too expensive."),
        ("E2",2,"I received a free ticket. I'm still not going because I have another commitment that evening.")],
     "candidate":"Sarah is not attending the conference because the ticket is too expensive."},
    {"id":"T2","evidence":[
        ("E1",1,"For this project, I want you to give me one recommendation."),
        ("E2",2,"I've moved into exploration mode now. Give me two contrasting approaches instead.")],
     "candidate":"Sarah currently wants one recommendation for this project."},

    {"id":"C1","evidence":[("E1",1,"When I postpone professional meetings, it's often because my schedule is uncertain.")],
     "candidate":"When Sarah postpones something, schedule uncertainty is a likely reason."},
    {"id":"C2","evidence":[("E1",1,"For scientific questions, I prefer seeing competing explanations before settling on one.")],
     "candidate":"Sarah prefers receiving multiple alternatives whenever she makes a decision."},

    {"id":"X1","evidence":[
        ("E1",1,"I'm delaying this partnership because I need more evidence."),
        ("E2",2,"The evidence is sufficient now. I'm waiting for procurement approval.")],
     "candidate":"Sarah tends to delay partnerships because she is cautious about making commitments."},
    {"id":"X2","evidence":[("E1",1,"I paused this creative project for two weeks because I need to finish a paid client project.")],
     "candidate":"Sarah tends to lose creative motivation when professional demands increase."},
]

EXPECTED_ORDER = ["S1","S2","S3","S4","S5","S6","S7","S8","E1","E2","T1","T2","C1","C2","X1","X2"]
assert [c["id"] for c in CASES] == EXPECTED_ORDER
assert len(CASES) == 16


In [ ]:
# Frozen private ground truth.
# Kept separate from prompt construction and never supplied to the model.
#
# Orthogonal component convention:
# Entailment evaluates substantive content while scope breadth is scored separately.
# Temporal validity is N/A where there is no relevant evidence update.

GROUND_TRUTH = {
"S1":{"decision":"ACCEPT","entailment":"PASS","temporal":"N/A","scope":"PASS","failure":"NONE"},
"S2":{"decision":"ACCEPT","entailment":"PASS","temporal":"N/A","scope":"PASS","failure":"NONE"},
"S3":{"decision":"ACCEPT","entailment":"PASS","temporal":"N/A","scope":"PASS","failure":"NONE"},
"S4":{"decision":"ACCEPT","entailment":"PASS","temporal":"N/A","scope":"PASS","failure":"NONE"},
"S5":{"decision":"ACCEPT","entailment":"PASS","temporal":"N/A","scope":"PASS","failure":"NONE"},
"S6":{"decision":"ACCEPT","entailment":"PASS","temporal":"N/A","scope":"PASS","failure":"NONE"},
"S7":{"decision":"ACCEPT","entailment":"PASS","temporal":"N/A","scope":"PASS","failure":"NONE"},
"S8":{"decision":"ACCEPT","entailment":"PASS","temporal":"N/A","scope":"PASS","failure":"NONE"},

"E1":{"decision":"REJECT","entailment":"FAIL","temporal":"N/A","scope":"PASS","failure":"ENTAILMENT"},
"E2":{"decision":"REJECT","entailment":"FAIL","temporal":"N/A","scope":"PASS","failure":"ENTAILMENT"},
"T1":{"decision":"REJECT","entailment":"PASS","temporal":"FAIL","scope":"PASS","failure":"TEMPORAL"},
"T2":{"decision":"REJECT","entailment":"PASS","temporal":"FAIL","scope":"PASS","failure":"TEMPORAL"},
"C1":{"decision":"REJECT","entailment":"PASS","temporal":"N/A","scope":"FAIL","failure":"SCOPE"},
"C2":{"decision":"REJECT","entailment":"PASS","temporal":"N/A","scope":"FAIL","failure":"SCOPE"},
"X1":{"decision":"REJECT","entailment":"FAIL","temporal":"FAIL","scope":"FAIL","failure":"MIXED"},
"X2":{"decision":"REJECT","entailment":"FAIL","temporal":"N/A","scope":"FAIL","failure":"MIXED"},
}

assert set(GROUND_TRUTH) == set(EXPECTED_ORDER)
assert sum(v["decision"]=="ACCEPT" for v in GROUND_TRUTH.values()) == 8
assert sum(v["decision"]=="REJECT" for v in GROUND_TRUTH.values()) == 8


In [ ]:
# Protocol snapshot saved before model execution
PROTOCOL = {
    "experiment": "Experiment 6: Decomposed Interpretation Verification",
    "phase": "A verification only",
    "model": MODEL_NAME,
    "seed": SEED,
    "do_sample": False,
    "case_order": EXPECTED_ORDER,
    "hv_prompt": HV_PROMPT,
    "dv_prompt": DV_PROMPT,
    "mechanical_dv_rule": "ACCEPT iff all applicable DV dimensions PASS; N/A is non-failing; any FAIL => REJECT",
    "repair_enabled": False,
    "behavior_probes_enabled": False,
}

with open(OUT_DIR/"protocol_snapshot.json","w") as f:
    json.dump(PROTOCOL, f, indent=2)

with open(OUT_DIR/"frozen_cases.json","w") as f:
    json.dump(CASES, f, indent=2)

with open(OUT_DIR/"private_ground_truth.json","w") as f:
    json.dump(GROUND_TRUTH, f, indent=2)

print("Protocol snapshot, frozen cases, and private ground truth saved.")


In [ ]:
# Safety checks: prompt construction must contain only neutral evidence + candidate.
FORBIDDEN_METADATA = ["superseded_as_current_reason", "status=current", "status=superseded"]

def evidence_block(case):
    return "\n".join(f"{eid} | turn {turn} | {statement}" for eid, turn, statement in case["evidence"])

def build_case_input(case):
    return f"EVIDENCE:\n{evidence_block(case)}\n\nCANDIDATE INTERPRETATION:\n{case['candidate']}"

def build_messages(base_prompt, case):
    case_input = build_case_input(case)
    low = case_input.lower()
    for forbidden in FORBIDDEN_METADATA:
        if forbidden.lower() in low:
            raise RuntimeError(f"Forbidden semantic metadata leaked into case {case['id']}: {forbidden}")
    return [
        {"role":"system","content":base_prompt},
        {"role":"user","content":case_input}
    ]

# Strict information parity check: user evidence/candidate text is byte-identical for HV and DV.
for case in CASES:
    hv_user = build_messages(HV_PROMPT, case)[1]["content"]
    dv_user = build_messages(DV_PROMPT, case)[1]["content"]
    assert hv_user == dv_user

print("Information parity checks passed for all 16 cases.")


In [ ]:
# Load model
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype="auto",
    device_map="auto"
)
model.eval()
print("Loaded:", MODEL_NAME)


In [ ]:
# Deterministic generation.
# Explicitly override the model repository's sampling defaults.

def generate(messages, max_new_tokens):
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([text], return_tensors="pt").to(model.device)
    input_len = inputs["input_ids"].shape[-1]

    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated = output[0][input_len:]
    hit_ceiling = len(generated) >= max_new_tokens
    decoded = tokenizer.decode(generated, skip_special_tokens=True).strip()
    return decoded, int(len(generated)), hit_ceiling


In [ ]:
# Run all 32 independent evaluations.
# RAW OUTPUTS ARE SAVED BEFORE ANY PARSING OR SCORING.

raw = []
start_all = time.time()

for i, case in enumerate(CASES, 1):
    print(f"[{i:02d}/16] {case['id']} HV...", flush=True)
    t0 = time.time()
    hv_text, hv_tokens, hv_ceiling = generate(build_messages(HV_PROMPT, case), MAX_NEW_TOKENS_HV)
    if hv_ceiling:
        raise RuntimeError(f"HV hard token ceiling reached on {case['id']}. Run invalid; stop before scoring.")

    print(f"[{i:02d}/16] {case['id']} DV...", flush=True)
    dv_text, dv_tokens, dv_ceiling = generate(build_messages(DV_PROMPT, case), MAX_NEW_TOKENS_DV)
    if dv_ceiling:
        raise RuntimeError(f"DV hard token ceiling reached on {case['id']}. Run invalid; stop before scoring.")

    raw.append({
        "case_id": case["id"],
        "evidence": case["evidence"],
        "candidate": case["candidate"],
        "hv_raw": hv_text,
        "dv_raw": dv_text,
        "hv_generated_tokens": hv_tokens,
        "dv_generated_tokens": dv_tokens,
        "elapsed_seconds": round(time.time()-t0, 3),
    })

    # Save after every completed pair so raw results exist independently of scoring.
    with open(OUT_DIR/"raw_outputs.json","w") as f:
        json.dump(raw, f, indent=2)

print("All raw outputs saved.")
print("Total seconds:", round(time.time()-start_all, 2))


In [ ]:
# Strict parsers. Invalid/missing/duplicated required labels => PARSE_FAILURE.

def unique_field(text, field, allowed):
    matches = re.findall(rf"(?mi)^\s*{re.escape(field)}\s*:\s*([^\n\r]+)\s*$", text)
    if len(matches) != 1:
        return None
    value = matches[0].strip().upper()
    return value if value in allowed else None

def free_field(text, field):
    matches = re.findall(rf"(?mi)^\s*{re.escape(field)}\s*:\s*([^\n\r]+)\s*$", text)
    return matches[0].strip() if len(matches)==1 else None

def parse_hv(text):
    decision = unique_field(text, "DECISION", {"ACCEPT","REJECT"})
    evidence = free_field(text, "EVIDENCE")
    justification = free_field(text, "JUSTIFICATION")
    ok = decision is not None and evidence is not None and justification is not None
    return {"parse_status":"OK" if ok else "PARSE_FAILURE",
            "decision":decision, "evidence":evidence, "justification":justification}

def parse_dv(text):
    entail = unique_field(text, "ENTAILMENT", {"PASS","FAIL","N/A"})
    temporal = unique_field(text, "TEMPORAL_VALIDITY", {"PASS","FAIL","N/A"})
    scope = unique_field(text, "SCOPE", {"PASS","FAIL","N/A"})
    final = unique_field(text, "FINAL_DECISION", {"ACCEPT","REJECT"})
    evidence = free_field(text, "EVIDENCE")
    justification = free_field(text, "JUSTIFICATION")
    ok = all(x is not None for x in [entail, temporal, scope, final, evidence, justification])
    return {"parse_status":"OK" if ok else "PARSE_FAILURE",
            "entailment":entail, "temporal":temporal, "scope":scope,
            "final_decision":final, "evidence":evidence, "justification":justification}

parsed = []
for r in raw:
    hp = parse_hv(r["hv_raw"])
    dp = parse_dv(r["dv_raw"])
    if hp["parse_status"] != "OK" or dp["parse_status"] != "OK":
        # Save parsing result, then terminate before scoring per frozen protocol.
        parsed.append({"case_id":r["case_id"],"hv":hp,"dv":dp})
        with open(OUT_DIR/"parsed_outputs.json","w") as f:
            json.dump(parsed, f, indent=2)
        raise RuntimeError(f"PARSE_FAILURE on {r['case_id']}. Frozen protocol requires stopping before scoring.")
    parsed.append({"case_id":r["case_id"],"hv":hp,"dv":dp})

with open(OUT_DIR/"parsed_outputs.json","w") as f:
    json.dump(parsed, f, indent=2)

print("All 32 outputs parsed successfully and saved. Scoring may now begin.")


In [ ]:
# Mechanical DV and frozen scoring

def mechanical_dv(dp):
    dims = [dp["entailment"], dp["temporal"], dp["scope"]]
    return "REJECT" if "FAIL" in dims else "ACCEPT"

rows = []
for item in parsed:
    cid = item["case_id"]
    gt = GROUND_TRUTH[cid]
    hv = item["hv"]
    dv = item["dv"]
    mech = mechanical_dv(dv)

    rows.append({
        "case_id": cid,
        "ground_truth": gt["decision"],
        "failure_type": gt["failure"],
        "hv_decision": hv["decision"],
        "dv_final": dv["final_decision"],
        "mechanical_dv": mech,
        "gt_entailment": gt["entailment"],
        "dv_entailment": dv["entailment"],
        "gt_temporal": gt["temporal"],
        "dv_temporal": dv["temporal"],
        "gt_scope": gt["scope"],
        "dv_scope": dv["scope"],
        "hv_correct": int(hv["decision"] == gt["decision"]),
        "dv_correct": int(dv["final_decision"] == gt["decision"]),
        "mechanical_correct": int(mech == gt["decision"]),
        "entailment_correct": int(dv["entailment"] == gt["entailment"]),
        "temporal_correct": int(dv["temporal"] == gt["temporal"]),
        "scope_correct": int(dv["scope"] == gt["scope"]),
    })

df = pd.DataFrame(rows)
df.to_csv(OUT_DIR/"case_scores.csv", index=False)
df


In [ ]:
# Summary metrics

def classification_metrics(pred_col):
    accept = df["ground_truth"] == "ACCEPT"
    reject = df["ground_truth"] == "REJECT"
    return {
        "accuracy": float((df[pred_col] == df["ground_truth"]).mean()),
        "sensitivity_corruption_rejection": float((df.loc[reject, pred_col] == "REJECT").mean()),
        "specificity_valid_acceptance": float((df.loc[accept, pred_col] == "ACCEPT").mean()),
        "false_rejection_rate": float((df.loc[accept, pred_col] == "REJECT").mean()),
        "false_acceptance_rate": float((df.loc[reject, pred_col] == "ACCEPT").mean()),
        "correct_n": int((df[pred_col] == df["ground_truth"]).sum()),
    }

summary = {
    "HV": classification_metrics("hv_decision"),
    "DV_model_composed": classification_metrics("dv_final"),
    "DV_mechanical": classification_metrics("mechanical_dv"),
    "DV_components": {
        "entailment_accuracy": float(df["entailment_correct"].mean()),
        "temporal_accuracy": float(df["temporal_correct"].mean()),
        "scope_accuracy": float(df["scope_correct"].mean()),
    }
}

with open(OUT_DIR/"summary_metrics.json","w") as f:
    json.dump(summary, f, indent=2)

print(json.dumps(summary, indent=2))


In [ ]:
# Failure localization for the 8 corrupted cases

def expected_failed_dims(gt):
    return {d for d, val in [
        ("ENTAILMENT", gt["entailment"]),
        ("TEMPORAL", gt["temporal"]),
        ("SCOPE", gt["scope"])
    ] if val == "FAIL"}

def observed_failed_dims(dv):
    return {d for d, val in [
        ("ENTAILMENT", dv["entailment"]),
        ("TEMPORAL", dv["temporal"]),
        ("SCOPE", dv["scope"])
    ] if val == "FAIL"}

localization = []
for item in parsed:
    cid = item["case_id"]
    gt = GROUND_TRUTH[cid]
    if gt["decision"] != "REJECT":
        continue
    exp = expected_failed_dims(gt)
    obs = observed_failed_dims(item["dv"])
    localization.append({
        "case_id": cid,
        "expected_failed_dimensions": sorted(exp),
        "observed_failed_dimensions": sorted(obs),
        "full_localization_correct": exp == obs,
        "all_expected_failures_detected": exp.issubset(obs),
    })

loc_df = pd.DataFrame(localization)
loc_df.to_csv(OUT_DIR/"failure_localization.csv", index=False)
loc_df


## Evidence citation scoring

Per the frozen protocol, evidence citation quality is a separate secondary measure:

- **1** relevant and sufficient
- **0.5** relevant but incomplete
- **0** irrelevant, contradictory, or no relevant evidence cited

This notebook does **not** automatically assign those semantic scores. Doing so would introduce a new model judge or an unfrozen heuristic. Score this column manually after the primary frozen results are saved.


In [ ]:
# Manual evidence scoring template, created only after primary outputs/scores exist.
evidence_template = pd.DataFrame([
    {
        "case_id": item["case_id"],
        "hv_evidence": item["hv"]["evidence"],
        "hv_evidence_score": "",
        "dv_evidence": item["dv"]["evidence"],
        "dv_evidence_score": "",
        "notes": ""
    }
    for item in parsed
])
evidence_template.to_csv(OUT_DIR/"manual_evidence_scoring_template.csv", index=False)
evidence_template


## Interpretation constraint

This is a 16-case synthetic evaluation with one deterministic model configuration.

Even a perfect result does not establish general architectural superiority.

Appropriate wording is:

> Under this controlled synthetic evaluation and model configuration, decomposed verification improved discrimination between evidence-supported and corrupted personalized interpretations relative to holistic verification.

If decomposition fails, the conclusion applies to the tested implementation rather than proving that decomposition cannot work.
